In [ ]:
# Load packages

using JuMP
using Mosek, MosekTools
using LinearAlgebra,OffsetArrays

WebIO._IJuliaInit()

Plots.PlotlyJSBackend()

In [2]:
# Linear algebra

function e_i(n, i)
    out = zeros(n, 1)
    out[i] = 1
    return out
end

function ⊙(a,b)
    return ((a*b') .+ transpose(a*b')) ./ 2
end

function sym(X)
    return (X + X')/2
end

function construct_R(n)
    return reverse(Matrix(1.0 * I,n,n), dims=2)
end

function construct_S(n)
    return [i>=j for i=1:n, j=1:n]
end

function construct_P(n)
    return construct_S(n) * construct_R(n)
end

function antitranspose(H)
    n = size(H)[1]
    R = construct_R(n)
    return R * H' * R
end

antitranspose (generic function with 1 method)

In [3]:
# FSFOMs

function tH_to_H(tH, q)
    N = size(tH)[1] - 1
    S = construct_S(N+1)
    
    hatH = tH + inv(S)
    H = hatH * inv(I + q * S * hatH)
    
    return H
end

function H_to_tH(H, q)
    N = size(H)[1] - 1
    S = construct_S(N+1)

    hatH = inv(I - q * H * S) * H
    tH = hatH - inv(S)

    return tH
end

function mu_to_q(mu)
    return mu/(1-mu)
end

function q_to_mu(q)
    return q/(1+q)
end

function construct_picard(N, q)
    mu = q_to_mu(q)
    tH = zeros(N+1,N+1)
    for i = 2:N+1
        tH[i,i-1] = 2/(1+mu)
    end
    return tH
end

function construct_ITEM(N, q)
    mu = q_to_mu(q)

    tH = zeros(N+1,N+1)
    Lambda = zeros(N+1,N+1)

    Aprev = 0

    x = zeros(N+1)
    yprev = zeros(N+1)
    z = zeros(N+1)

    for i = 0:N-1
        A = ((1+mu) * Aprev + 2 * (1 + sqrt((1 + Aprev) * (1 + mu * Aprev)))) / (1-mu)^2
        beta = Aprev / ((1-mu) * A)
        delta = ((1-mu)^2 * A - (1+mu) * Aprev) / (2 * (1 + mu + mu * Aprev))

        y = (1 - beta) * z + beta * x
        x = y - e_i(N+1,i+1)
        z = (1 - mu * delta) * z + mu * delta * y - delta * e_i(N+1, i+1)

        tH[i+1,:] = yprev - y
        Aprev = A
        yprev = y

        Lambda[i+1,i+1] -= A
        if i < N - 1
            Lambda[i+2,i+1] += A
        end
    end
    tH[N+1,:] = yprev - z

    tau = 1 / sqrt(1 + mu * Aprev)
    Lambda .*= 0.5 * tau * (1-mu)
    return tH, Lambda
end

function construct_OC_Halpern(N, q)
    gamma = 1.0 + 2 * q
    mu = q / (1 + q)

    phi = 1.0
    gamma_power_squared = 1.0

    tH = zeros(N+1, N+1)
    Lambda = zeros(N+1, N+1)

    yprev = zeros(N+1)

    geometric_sum = sum(gamma^i for i = 0:N)
    tau = 1 / geometric_sum

    terminal_power = gamma^(N+1)
    certificate_scale =
        tau * terminal_power / (1 + terminal_power)

    for i = 1:N
        phi_prev = phi

        gamma_power_squared *= gamma^2
        phi += gamma_power_squared

        averaging = 1 - inv(phi)

        y = averaging * (
            yprev - 2 / (1 + mu) * e_i(N+1, i)
        )

        tH[i+1, :] = yprev - y
        yprev = y

        weight =
            certificate_scale *
            (1 + gamma) *
            phi *
            phi_prev /
            gamma^(2i)

        Lambda[i, i] -= weight
        Lambda[i+1, i+1] -= weight
        Lambda[i, i+1] += weight
        Lambda[i+1, i] += weight
    end

    # Interpolation constraint between the final point and the fixed point.
    Lambda[N+1, N+1] -= 1

    return tH, Lambda
end

construct_OC_Halpern (generic function with 1 method)

In [4]:
# Performance Estimation Problems

function pep_fixed_H(H, q, M0, MN; symmetric=false, diagonal=false)
    # Assume M0 is a 2x2 PSD matrix.
    # MN is either an invertible 2x2 matrix
    # or it is a vector u representing MN = uu'

    N = size(H)[2]-1
    onevec = ones(N+1)
    S = construct_S(N+1)
    e0, eN = e_i(N+1,1), e_i(N+1,N+1)

    opt_model = Model(optimizer_with_attributes(Mosek.Optimizer))

    if diagonal
        @variable(opt_model, lambda[1:N+1])
        Lambda = diagm(lambda)
    elseif symmetric
        @variable(opt_model, Lambda[1:N+1,1:N+1], Symmetric)
    else
        @variable(opt_model, Lambda[1:N+1,1:N+1])
    end

    @variable(opt_model, tau >= 0)

    @objective(opt_model, Min, tau)

    vH = (I - q * S * H) * onevec

    certificate_B11 = 0
    certificate_B12 = - vH' * Lambda'
    certificate_B22 = 2 * sym(Lambda * S * H)
    certificate = [certificate_B11 certificate_B12
        certificate_B12' certificate_B22]

    y0_coeff = vcat(1.0, zeros(N+1))
    s0_coeff = vcat(
        q / (1+q),
        1/(1+q) * e0
    )

    sN_coeff = vcat(
        q * dot(eN, vH),
        (I - q * S * H)' * eN
    )
    yN_coeff = vcat(
        dot(eN,vH),
        (- S * H)' * eN
    ) + sN_coeff

    E0 = [y0_coeff s0_coeff]
    EN = [yN_coeff sN_coeff]

    A = tau * E0 * M0 * E0' - certificate

    if length(MN) == 4
        slack = [
            A EN
            EN' (tau * inv(MN))
        ]           
    else
        u = MN
        c = EN * u
        slack = [
            A c
            c' tau
        ]
    end
    
    @constraint(opt_model,
    slack >= 0, PSDCone())

    @constraint(opt_model, [i=1:N+1,j=1:N+1;i!=j], Lambda[i,j] >= 0)
    @constraint(opt_model, Lambda * onevec <= 0)
    @constraint(opt_model, Lambda' * onevec <= 0)

    set_silent(opt_model)

    optimize!(opt_model)

    if termination_status(opt_model) != MOI.OPTIMAL
        @warn "opt_model solving did not reach optimality;  termination status = " termination_status(opt_model)
    end

    return value(tau), value.(Lambda)
end

function pep_fo_H(H, Lambda, q, M0, MN; symmetric = false, diagonal=false, bounds = 1e-1)
    N = size(H)[2]-1
    onevec = ones(N+1)
    S = construct_S(N+1)
    e0, eN = e_i(N+1,1), e_i(N+1,N+1)

    opt_model = Model(optimizer_with_attributes(Mosek.Optimizer))

    if diagonal
        @variable(opt_model, dlambda[1:N+1])
        dLambda = diagm(dlambda)
    elseif symmetric
        @variable(opt_model, dLambda[1:N+1,1:N+1], Symmetric)
    else
        @variable(opt_model, dLambda[1:N+1,1:N+1])
    end
 
    @variable(opt_model, dH[1:N+1,1:N+1])

    for i = 1:N+1
        for j = i:N+1
            fix(dH[i,j], 0.0; force=true)
        end
    end


    @variable(opt_model, tau >= 0)

    @objective(opt_model, Min, tau)

    vH = (I - q * S * H) * onevec
    dvH = - q * S * dH * onevec

    certificate_B11 = 0
    certificate_B12 = - vH' * Lambda' - dvH' * Lambda' - vH' * dLambda'
    certificate_B22 = 2 * sym(Lambda * S * H) + 2 * sym(dLambda * S * H) + 2 * sym(Lambda * S * dH)
    certificate = [certificate_B11 certificate_B12
        certificate_B12' certificate_B22]

    y0_coeff = vcat(1.0, zeros(N+1))
    s0_coeff = vcat(
        q/(1+q),
        1/(1+q) * e0
    )

    sN_coeff = vcat(
        q * dot(eN, vH + dvH),
        (I - q * S * (H + dH))' * eN
    )
    yN_coeff = vcat(
        dot(eN,vH + dvH),
        (- S * (H + dH))' * eN
    ) + sN_coeff

    E0 = [y0_coeff s0_coeff]
    EN = [yN_coeff  sN_coeff]
   
    A = tau * E0 * M0 * E0' - certificate

    if length(MN) == 4
        slack = [
            A EN
            EN' (tau * inv(MN))
        ]           
    else
        u = MN
        c = EN * u
        slack = [
            A c
            c' tau
        ] 
    end

    @constraint(opt_model,
    slack >= 0, PSDCone())

    @constraint(opt_model, [i=1:N+1,j=1:N+1;i!=j], Lambda[i,j] + dLambda[i,j] >= 0)
    @constraint(opt_model, (Lambda + dLambda) * onevec <= 0)
    @constraint(opt_model, (Lambda + dLambda)' * onevec <= 0)

    @constraint(opt_model, -bounds .<= dLambda .<= bounds)
    @constraint(opt_model, -bounds .<= dH .<= bounds)

    set_silent(opt_model)

    optimize!(opt_model)

    if termination_status(opt_model) != MOI.OPTIMAL
        @warn "opt_model solving did not reach optimality;  termination status = " termination_status(opt_model)
    end

    return value(tau), value.(dH)
end

function guess_opt(q, N, M0, MN; symmetric=false, diagonal=false)
    S = construct_S(N+1)
    tH = randn(N+1, N+1) .* (S - I)
    hatH = tH + inv(S)
    H = hatH * inv(I + q * S * hatH)

    for _ = 1:1000
        tau, Lambda = pep_fixed_H(H, q, M0, MN; symmetric=symmetric, diagonal=diagonal)
        _, dH = pep_fo_H(H, Lambda, q, M0, MN; symmetric=symmetric, diagonal=diagonal)
        H += dH
    end
    tau, Lambda = pep_fixed_H(H, q, M0, MN; symmetric=symmetric, diagonal=diagonal)
    return H, tau, Lambda
end

guess_opt (generic function with 1 method)

In [134]:
function h_dual_fo_search_DG(H, Lambda, Phi, q, tau; symmetric = false, diagonal=false, regularization = 1e-5)
    N = size(H)[2]-1
    onevec = ones(N+1)
    S = construct_S(N+1)
    e0, eN = e_i(N+1,1), e_i(N+1,N+1)

    opt_model = Model(optimizer_with_attributes(Mosek.Optimizer))

    if diagonal
        @variable(opt_model, dlambda[1:N+1])
        dLambda = diagm(dlambda)
        @variable(opt_model, dphi[1:N+1])
        dPhi = diagm(dphi)
    elseif symmetric
        @variable(opt_model, dLambda[1:N+1,1:N+1], Symmetric)
        @variable(opt_model, dPhi[1:N+1,1:N+1], Symmetric)
    else
        @variable(opt_model, dLambda[1:N+1,1:N+1])
        @variable(opt_model, dPhi[1:N+1,1:N+1])
    end

    P = construct_P(N+1)
    HA = antitranspose(H)
    vH = (I - q * S * H) * onevec
    vHA = (I - q * S * HA) * onevec
    eta = dot(eN, vH)

    error1 = P * Lambda * P * Phi + P * dLambda * P * Phi + P * Lambda * P * dPhi - I
    error2 = P * Phi * P * Lambda + P * dPhi * P * Lambda + P * Phi * P * dLambda - I

    @variable(opt_model, t >= 0)

    objective_vector = vcat(
        vec(error1),
        vec(error2),
        sqrt(regularization) .* vec(dLambda),
        sqrt(regularization) .* vec(dPhi)
    )

    @constraint(
        opt_model,
        vcat(t, 0.5, objective_vector)
            in RotatedSecondOrderCone()
    )

    @objective(opt_model, Min, t)

    slack_B11 = -2 * sym((Lambda + dLambda) * S * H)
    slack_B12 = (Lambda + dLambda) * vH
    slack_B13 = (I - q * S * H)' * eN
    slack_B22 = tau
    slack_B23 = q * eta
    slack_B33 = tau

    slack_H = [slack_B11 slack_B12 slack_B13
    slack_B12' slack_B22 slack_B23
    slack_B13' slack_B23' slack_B33]

    @constraint(opt_model,
    slack_H >= 0, PSDCone())

    @constraint(opt_model, [i=1:N+1,j=1:N+1;i!=j], Lambda[i,j] + dLambda[i,j] >= 0)
    @constraint(opt_model, (Lambda + dLambda) * onevec <= 0)
    @constraint(opt_model, (Lambda + dLambda)' * onevec <= 0)

    slack_B11 = -2 * sym((Phi + dPhi) * S * HA)
    slack_B12 = (Phi + dPhi) * vHA
    slack_B13 = (I - q * S * HA)' * eN
    slack_B22 = tau
    slack_B23 = q * eta
    slack_B33 = tau

    slack_HA = [slack_B11 slack_B12 slack_B13
    slack_B12' slack_B22 slack_B23
    slack_B13' slack_B23' slack_B33]

    @constraint(opt_model,
    slack_HA >= 0, PSDCone())

    @constraint(opt_model, [i=1:N+1,j=1:N+1;i!=j], Phi[i,j] + dPhi[i,j] >= 0)
    @constraint(opt_model, (Phi + dPhi) * onevec <= 0)
    @constraint(opt_model, (Phi + dPhi)' * onevec <= 0)

    set_silent(opt_model)

    optimize!(opt_model)

    if termination_status(opt_model) != MOI.OPTIMAL
        @warn "opt_model solving did not reach optimality;  termination status = " termination_status(opt_model)
    end

    return value.(dLambda), value.(dPhi)
end

function verify_DG(H, Lambda, q, tau)
    N = size(H)[2]-1
    onevec = ones(N+1)
    S = construct_S(N+1)
    e0, eN = e_i(N+1,1), e_i(N+1,N+1)

    vH = (I - q * S * H) * onevec
    eta = dot(eN, vH)
    
    slack_B11 = -2 * sym(Lambda * S * H)
    slack_B12 = Lambda * vH
    slack_B13 = (I - q * S * H)' * eN
    slack_B22 = tau
    slack_B23 = q * eta
    slack_B33 = tau

    slack = [slack_B11 slack_B12 slack_B13
    slack_B12' slack_B22 slack_B23
    slack_B13' slack_B23' slack_B33]

    offdiag = ones(N+1,N+1) - I
    return max(
        0.0,
        maximum(Lambda * onevec),
        maximum(Lambda' * onevec),
        maximum(offdiag .* (-Lambda)),
        -minimum(eigen(slack).values)
    )
end

function pair_error_DG(Lambda, Phi, P)
    return max(
        maximum(abs.(P * Lambda * P * Phi - I)),
        maximum(abs.(P * Phi * P * Lambda - I)),
    )
end

function refine_h_dual_pair_DG(H, Lambda, Phi, q, tau;
    symmetric=false, target=1e-5, max_iterations=150, patience=5)

    n = size(H, 1)
    P = construct_P(n)
    HA = antitranspose(H)

    pair_error = pair_error_DG(Lambda, Phi, P)
    stalled_iterations = 0

    for iteration = 1:max_iterations
        Lambda_error = verify_DG(H, Lambda, q, tau)
        Phi_error = verify_DG(HA, Phi, q, tau)

        if max(Lambda_error, Phi_error, pair_error) <= target
            return Lambda, Phi
        end

        old_pair_error = pair_error

        dLambda, dPhi = h_dual_fo_search_DG(H, Lambda, Phi, q,
            tau * (1 + target); symmetric=symmetric
        )

        # Backtracking line search.
        alpha = 1.0
        step_accepted = false

        while alpha >= 2.0^-10
            Lambda_candidate = Lambda + alpha * dLambda
            Phi_candidate = Phi + alpha * dPhi

            candidate_error =
                pair_error_DG(Lambda_candidate, Phi_candidate, P)

            if candidate_error < pair_error
                Lambda = Lambda_candidate
                Phi = Phi_candidate
                pair_error = candidate_error
                step_accepted = true
                break
            end

            alpha /= 2
        end

        relative_progress =
            (old_pair_error - pair_error) /
            max(old_pair_error, eps(Float64))

        if !step_accepted || relative_progress < 0.01
            stalled_iterations += 1
        else
            stalled_iterations = 0
        end

        # Escape only when progress has stalled.
        if stalled_iterations >= patience && pair_error > target
            PLambdaP = P * Lambda * P

            if cond(PLambdaP) < 1e12
                current_total_error = max(
                    pair_error,
                    Lambda_error,
                    Phi_error,
                )

                Phi_reset = inv(PLambdaP)

                reset_Phi_error = verify_DG(HA, Phi_reset, q, tau)
                reset_pair_error = pair_error_DG(Lambda, Phi_reset, P)

                reset_total_error = max(
                    reset_pair_error,
                    Lambda_error,
                    reset_Phi_error,
                )

                if reset_total_error <= current_total_error
                    Phi = Phi_reset
                    pair_error = pair_error_DG(Lambda, Phi, P)
                end
            end

            stalled_iterations = 0
        end
    end

    return Lambda, Phi
end

refine_h_dual_pair_DG (generic function with 1 method)

In [7]:
# Experiment helpers

function maximum_sign_violation(Lambda)
    n = size(Lambda)[1]
    onevec = ones(n)
    offdiag = ones(n,n) - I
    return max(
        0.0,
        maximum(Lambda * onevec),
        maximum(Lambda' * onevec),
        maximum(offdiag .* (-Lambda))
    )
end

maximum_sign_violation (generic function with 1 method)

# Non-standard duality

In [8]:
q = 0.1
mu = q/(1+q)
rho = sqrt((1+mu^2)/2)

N = 5

M0 = [1/2 0
0 1/2]
MN = [1
0]

M0p = [(rho-mu)^2 ((rho-mu)* (1-rho))
((rho-mu)* (1-rho)) (1-rho)^2]
MNp = [(1+mu)
-2]

S = construct_S(N+1)

for symmetric in [false, true]
    trials = 1000
    h_dual_succeses = 0
    symmetric = false

    for _=1:trials
        tH = (I - inv(S)) + 0.1 * (2 .* rand(N+1,N+1) .- 1) .* (S - I)
        H = tH_to_H(tH,q)

        rate1 = pep_fixed_H(H, q, M0, MN; symmetric=symmetric)[1]
        rate2 = pep_fixed_H(antitranspose(H), q, M0p, MNp; symmetric=symmetric)[1]

        err = abs(rate1-rate2)/min(rate1,rate2)
        if err < 1e-5
            h_dual_succeses += 1
        end
    end
    println(h_dual_succeses, " / ", trials)
end

1000 / 1000
1000 / 1000


# Non-tridiagonal certificates

In [139]:
h_dual_trials = 100
max_trials = 10000

N = 5
q = 0.1
tau_tolerance = 1e-7

# sample tH where first subdiagonal is uniform [0, coefficient] and remaining entries are uniform [-coefficient, coefficient]
coefficients = [
    0   0   0   0   0 0
    2   0   0   0   0 0
    1/2 2   0   0   0 0
    1/4 1/2 2   0   0 0
    1/8 1/4 1/2 2   0 0
    1/16 1/8 1/4 1/2 2 0 
]

for symmetric in [false, true]
    # experiment initialization

    H_dual_pairs = 0

    P = construct_P(N+1)

    M0 = [1 0; 0 0]
    MN = [0; 1]

    tH = nothing

    Lambda_errors = []
    Phi_errors = []
    pair_errors = []

    for _ = 1:max_trials
        tH = 2 * rand(N+1,N+1) .- 1
        tH[diagind(tH, -1)] .= rand(N)
        tH .*= coefficients

        H = tH_to_H(tH, q)
        HA = antitranspose(H)

        tau_H, Lambda = pep_fixed_H(H, q, M0, MN; symmetric=symmetric)
        tau_HA, Phi = pep_fixed_H(HA, q, M0, MN; symmetric=symmetric)

        if abs(tau_H - tau_HA) / max(tau_H, tau_HA) >= tau_tolerance
            continue
        end
        H_dual_pairs += 1
        tau = max(tau_H, tau_HA)

        pair_error = max(
            maximum(abs.(P * Lambda * P * Phi - I)),
            maximum(abs.(P * Phi * P * Lambda - I))
        )

        out = refine_h_dual_pair_DG(H, Lambda, Phi, q, tau; symmetric=symmetric)
        Lambda = out[1]
        Phi = out[2]
        
        Lambda_error = verify_DG(H, Lambda, q, tau)
        Phi_error = verify_DG(HA, Phi, q, tau)
        pair_error = max(
            maximum(abs.(P * Lambda * P * Phi - I)),
            maximum(abs.(P * Phi * P * Lambda - I))
        )

        push!(Lambda_errors, Lambda_error)
        push!(Phi_errors, Phi_error)
        push!(pair_errors, pair_error)

        if H_dual_pairs == h_dual_trials
            break
        end
    end

    max_errors = max.(pair_errors, Lambda_errors, Phi_errors)
    successes = sum(max_errors .< 1e-5)
    println(successes, "/", length(max_errors))
end


93/100


┌ Warning: opt_model solving did not reach optimality;  termination status = 
│   termination_status(opt_model) = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Main In[134]:89


92/100


# Experiment: (D,D)--(G,G) functions

In [599]:
q = 0.1
mu = q_to_mu(q)
N = 5

M0 = [1 0
0 0]
MN = [1
0]

barP = construct_P(N)

tH, LambdaV = construct_ITEM(N,q)
Lambda = LambdaV[1:N,1:N]

Phi = inv(barP * Lambda * barP)

println(maximum_sign_violation(Phi))

2.7755575615628914e-17


# Experiment (D,G)--(D,G) functions

In [ ]:
q = 0.1
N = 5

M0 = [1 0
0 0]
MN = [0
1]

P = construct_P(N+1)

H, tau, Lambda = guess_opt(q, N, M0, MN)

Phi = inv(P * Lambda * P)

println(maximum_sign_violation(Phi))

6.524637223033472e-8


# Experiment (G,D)--(G,D) functions

In [ ]:
q = 0.1
N = 5

M0 = [0 0
0 1]
MN = [1
0]

barP = construct_P(N)
barR = construct_R(N)

H, tau, LambdaW = guess_opt(q, N, M0, MN)

Lambda = barR * LambdaW[1:N,1:N] * barR
Phi = inv(barP * Lambda * barP)

println(maximum_sign_violation(Phi))

7.024662579976783e-9


# Experiment: (D,D)--(G,G) operators

In [ ]:
q = 0.1
N = 5

M0 = [1 0
0 0]
MN = [1
0]

barP = construct_P(N)

tH = construct_picard(N,q)
H = tH_to_H(tH, q)
tau, LambdaV = pep_fixed_H(H, q, M0, MN)

Lambda = LambdaV[1:N,1:N]
Phi = inv(barP * Lambda * barP)

println(maximum_sign_violation(Phi))

3.795522176455311e-8


# Experiment: (D,G)--(D,G) operators

In [ ]:
q = 0.1
N = 5

M0 = [1 0
0 0]
MN = [0
1]

P = construct_P(N+1)

tH, Lambda = construct_OC_Halpern(N,q)

Phi = inv(P * Lambda * P)

println(maximum_sign_violation(Phi))

5.778474753458938e-16


# Experiment: (G,D)--(G,D) operators

In [ ]:
q = 0.1
N = 5

M0 = [0 0
0 1]
MN = [1
0]

barP = construct_P(N)
barR = construct_R(N)

H, tau, LambdaW = guess_opt(q, N, M0, MN; symmetric=true, diagonal=true)

Lambda = barR * LambdaW[1:N,1:N] * barR
Phi = inv(barP * Lambda * barP)

println(maximum_sign_violation(Phi))

4.894535363093433e-16
